# Administrative boundaries — catalog explorer (offline)

Before fetching anything, it helps to know **which datasets exist** and **which
selector each needs**. This notebook is fully offline — it only reads the bundled
catalog. By the end you will be able to list every admin dataset id and read its
provider, ADM level / layer, native CRS, and required selectors.

## Setup

The admin `Catalog` loads the bundled per-provider YAML files.

In [ ]:
import pandas as pd

from earthlens.admin import Catalog

cat = Catalog()
len(cat)

## Every dataset, as a table

Each row carries its `provider`, the ADM level (geoBoundaries / CGAZ) or layer
(Natural Earth / TIGER), the native CRS, and the selectors the request must supply.

In [ ]:
rows = [
    {
        'id': d.id,
        'provider': d.provider,
        'adm_level': d.adm_level,
        'layer': d.layer,
        'native_crs': d.native_crs,
        'required_selectors': ', '.join(d.required_selectors) or '—',
    }
    for d in cat.datasets.values()
]
pd.DataFrame(rows).sort_values('id').reset_index(drop=True)

## Datasets per provider

Four providers, each with a different addressing model — per-country ADM levels
(geoBoundaries), seamless global levels (CGAZ), scaled cultural layers
(Natural Earth), and US entities (TIGER).

In [ ]:
by_provider = {}
for d in cat.datasets.values():
    by_provider.setdefault(d.provider, []).append(d.id)
for provider, ids in sorted(by_provider.items()):
    print(f'{provider:14} {len(ids):>2}  {sorted(ids)}')

## Look up one dataset

`Catalog.get(id)` returns the typed row; an unknown id raises a clear `ValueError`
with a did-you-mean hint.

In [ ]:
row = cat.get('tiger:tract')
print('provider :', row.provider)
print('layer    :', row.layer)
print('per_state:', row.per_state)
print('selectors:', row.required_selectors)
print('license  :', row.license_note)

A typo is rejected rather than silently returning nothing:

In [ ]:
try:
    cat.get('geoboundaries:adm9')
except ValueError as exc:
    print(exc)

## Takeaway

- `Catalog()` lists 15 datasets across geoBoundaries, CGAZ, Natural Earth, TIGER.
- Each row tells you the selector to pass (`country=` / `scale=` / `year=` /
  `state=`) and the native CRS (the backend normalises every result to EPSG:4326).
- **GADM is intentionally absent** — its no-commercial / no-redistribute license is
  incompatible; the four shipped sources are CC-BY-4.0 or public domain.